In [ ]:
import numpy as np
from sklearn.neighbors import KernelDensity
from scipy.spatial import cKDTree
import geopandas as gpd
from shapely.geometry import Point

# KDE Parameters
bandwidth = 0.005  
lon_range = 0.05
lat_range = 0.05
num_grid_points = 500  

def calculate_kde_and_distances_in_meters(gdf, transit_stops, center_lon, center_lat):
    # Ensure geometries are valid
    gdf = gdf[gdf.geometry.notnull()]
    
    if gdf.empty:
        print("No data points to process.")
        return None

    # Extract geometry points
    coords = np.array([[point.x, point.y] for point in gdf.geometry])
    
    if coords.shape[0] == 0:
        print("No coordinates to process.")
        return None

    # Perform kde
    kde = KernelDensity(bandwidth=bandwidth, kernel='gaussian')
    kde.fit(coords)
    
    # Create a grid to evaluate the KDE
    x_min_zoomed = center_lon - lon_range / 2
    x_max_zoomed = center_lon + lon_range / 2
    y_min_zoomed = center_lat - lat_range / 2
    y_max_zoomed = center_lat + lat_range / 2
    
    x_grid = np.linspace(x_min_zoomed, x_max_zoomed, num_grid_points)
    y_grid = np.linspace(y_min_zoomed, y_max_zoomed, num_grid_points)
    x_mesh, y_mesh = np.meshgrid(x_grid, y_grid)
    xy_sample = np.vstack([x_mesh.ravel(), y_mesh.ravel()]).T
    z = np.exp(kde.score_samples(xy_sample)).reshape(x_mesh.shape)
    
    # Identify high-density areas 
    threshold = np.percentile(z, 90)  # Top 10% density
    hotspot_indices = np.argwhere(z > threshold)

    # Convert grid indices to geographic coordinates
    hotspots = [
        Point(x_grid[j], y_grid[i])
        for i, j in hotspot_indices
    ]
    hotspots_gdf = gpd.GeoDataFrame(geometry=hotspots, crs=gdf.crs)

    # Re-project hotspots and transit stops to a meter-based CRS (EPSG:3857)
    projected_crs = 'EPSG:3857'  # CRS using meters
    hotspots_gdf = hotspots_gdf.to_crs(projected_crs)
    transit_stops = transit_stops.to_crs(projected_crs)

    # Extract coordinates for nearest neighbor calculation
    hotspot_coords = np.array([[geom.x, geom.y] for geom in hotspots_gdf.geometry])
    transit_coords = np.array([[geom.x, geom.y] for geom in transit_stops.geometry])

    # Build a KDTree for the transit stops
    transit_tree = cKDTree(transit_coords)

    # Query the nearest transit stop for each hotspot
    distances, indices = transit_tree.query(hotspot_coords)

    # Add distances to the hotspots GeoDataFrame
    hotspots_gdf['distance_to_nearest_stop_meters'] = distances

    return hotspots_gdf


# Load homelessness data 
homelessness_gdf = gpd.read_file('../data/2022/hierarchical_data_2022.geojson')

# Load transit stops data 
transit_stops_gdf = gpd.read_file('../data/Public_Transit_Stops%2C_San_Diego_County.geojson')

# Define city center coordinates 
center_lon = -117.1611  
center_lat = 32.7157   

# Calculate KDE and distances in meters
hotspots_with_distances = calculate_kde_and_distances_in_meters(homelessness_gdf, transit_stops_gdf, center_lon, center_lat)

# Display the first few rows
hotspots_with_distances.head()


In [ ]:
# Calculate the average distance to the nearest transit stop
average_distance = hotspots_with_distances['distance_to_nearest_stop_meters'].mean()

# Print the average distance
print(f"The average distance to the nearest transit stop is {average_distance:.2f} meters.")


In [ ]:
transit_stops_gdf.shape

In [ ]:
import geopandas as gpd

transit_stops_gdf = gpd.read_file('/Users/kaelananderson/Documents/HDMA/311 Data/data/Public_Transit_Stops%2C_San_Diego_County.geojson')


In [ ]:
transit_stops_gdf

In [ ]:
location_type_counts = transit_stops_gdf['location_type'].value_counts()
print(location_type_counts)

In [ ]:
filtered_df = transit_stops_gdf[transit_stops_gdf['location_type'] == '2']
filtered_df